In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval[onnx,opencv] maite-datasets torchvision
except Exception:
    pass

In [2]:
import warnings
from collections.abc import Callable
from pathlib import Path

import numpy as np
import torch
from maite_datasets.object_detection import VOCDetection
from torchvision.models import ResNet18_Weights, resnet18

from dataeval import Embeddings
from dataeval.data import Limit, Resize, View
from dataeval.extractors import BoVWExtractor, FlattenExtractor, OnnxExtractor, TorchExtractor
from dataeval.utils.onnx import to_encoding_model

In [3]:
dataset = VOCDetection(root="./data", image_set="val", year="2012", download=True)
subset = View(dataset, Limit(100))

print(f"Images: {len(subset)}")
print(f"Shape of the first image: {subset[0][0].shape}")

Images: 100
Shape of the first image: (3, 375, 500)


In [4]:
weights = ResNet18_Weights.DEFAULT
torch_extractor = TorchExtractor(
    resnet18(weights=weights, progress=False),
    transforms=weights.transforms(),
    layer_name="avgpool",
)
torch_embeddings = Embeddings(subset, extractor=torch_extractor, batch_size=32)

print(f"Embeddings shape: {torch_embeddings.shape}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


Embeddings shape: (100, 512)


In [5]:
model_path = Path("data") / "resnet18.onnx"
model_path.parent.mkdir(exist_ok=True)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    torch.onnx.export(
        resnet18(weights=weights, progress=False).eval(),
        (torch.zeros(1, 3, 224, 224),),
        str(model_path),
        input_names=["image"],
        output_names=["logits"],
        dynamic_axes={"image": {0: "batch"}, "logits": {0: "batch"}},
        opset_version=13,
        dynamo=False,
    )

In [6]:
encoding_model, embedding_layer = to_encoding_model(model_path)

print(f"Embedding layer: {embedding_layer}")

Embedding layer: /Flatten_output_0


In [7]:
def numpy_transform(transform: Callable[[torch.Tensor], torch.Tensor]) -> Callable[[np.ndarray], np.ndarray]:
    """Run a torchvision preprocessing transform on one NumPy image and return a NumPy array."""

    def apply(image: np.ndarray) -> np.ndarray:
        return transform(torch.as_tensor(np.asarray(image))).numpy()

    return apply


onnx_extractor = OnnxExtractor(
    encoding_model,
    transforms=numpy_transform(weights.transforms()),
    output_name=embedding_layer,
)
onnx_embeddings = Embeddings(subset, extractor=onnx_extractor, batch_size=32)

print(f"Embeddings shape: {onnx_embeddings.shape}")

/tmp/ipykernel_4839/2908367292.py:5: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return transform(torch.as_tensor(np.asarray(image))).numpy()


Embeddings shape: (100, 512)


In [8]:
# TEST ASSERTION CELL ###
# the ONNX path reproduces the PyTorch embeddings
torch_array = np.asarray(torch_embeddings)
onnx_array = np.asarray(onnx_embeddings)
cosine = (torch_array * onnx_array).sum(axis=1) / (
    np.linalg.norm(torch_array, axis=1) * np.linalg.norm(onnx_array, axis=1)
)
assert torch_array.shape == onnx_array.shape == (100, 512)
assert cosine.min() > 0.999

In [9]:
resized = View(subset, Resize((128, 128)))

flatten_embeddings = Embeddings(resized, extractor=FlattenExtractor(), batch_size=32)
bovw_extractor = BoVWExtractor(vocab_size=64).fit(resized)
bovw_embeddings = Embeddings(resized, extractor=bovw_extractor, batch_size=32)

print(f"Flatten embeddings shape: {flatten_embeddings.shape}")
print(f"BoVW embeddings shape: {bovw_embeddings.shape}")

Flatten embeddings shape: (100, 49152)
BoVW embeddings shape: (100, 64)


In [10]:
# TEST ASSERTION CELL ###
assert flatten_embeddings.shape == (100, 3 * 128 * 128)
assert bovw_embeddings.shape == (100, 64)